# Home Health Agency Quality Analysis
This project analyzes CMS data to identify what drives quality in Home Health Care using the CRISP-DM process.

## CRISP-DM Phase 1: Business Understanding
I aim to answer the following four questions:
1. **Ownership Impact:** Do non-profit agencies have higher star ratings than for-profits?
2. **Clinical Outcomes:** Do non-profits achieve better clinical results (Mobility Improvement)?
3. **Financial Efficiency:** Which ownership type has the best Medicare Spend Ratio?
4. **Predictive Modeling:** Which clinical metrics are the strongest predictors of an agency's Star Rating?

## CRISP-DM Phase 2: Data Understanding
In this section, I load the raw CMS dataset and perform Exploratory Data Analysis (EDA) to understand the distribution of variables and identify the extent of missing data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# Access: Load the dataset
df_raw = pd.read_csv('HH_Provider_Jan2026.csv')

# Explore: Check dimensions and data types
print(f"Dataset contains {df_raw.shape[0]} rows and {df_raw.shape[1]} columns.")
display(df_raw.head())
print(df_raw.info())

# Explore: Analyze missing values
null_counts = df_raw[['Quality of Patient Care Star Rating', 'Type of Ownership']].isnull().sum()
print("\nMissing values in key columns:")
print(null_counts)

# Explore: Visualize the raw distribution of the target variable
plt.figure(figsize=(6,4))
sns.countplot(data=df_raw, x='Quality of Patient Care Star Rating')
plt.title('Distribution of Raw Star Ratings')
plt.show()

### Handling Missing Values Justification
During the exploration phase, I identified significant missing data in the Star Rating and Clinical Outcome columns.

1. **Dropping Rows:** I chose to **drop** rows where the `Quality of Patient Care Star Rating` is missing. As this is the target variable for my predictive model, imputing it would introduce artificial bias and decrease the reliability of the analysis.
2. **Imputing Clinical Metrics:** For clinical columns like `Mobility Improvement`, I opted for **Mean Imputation**. Because the missing data in these columns represents a relatively small portion of the total dataset and the values are numeric, using the mean allows me to retain the agencies for analysis without significantly altering the feature's distribution.

## CRISP-DM Phase 3: Data Preparation
I use a modular function to clean the dataset. This ensures that the cleaning process is reproducible, robust, and well-documented.

In [ ]:
def clean_cms_data(df):
    """
    Description:
        Cleans the raw CMS dataframe by standardizing column names to PEP8 standards, 
        converting string types to numeric floats, and handling missing values.
    
    Arguments:
        df (pandas DataFrame): The raw DataFrame loaded from the CSV.
        
    Returns:
        df_prepared (pandas DataFrame): The processed and cleaned DataFrame.
    """
    # 1. Clean and standardize column names (snake_case)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '').str.replace('/', '_').str.replace(',', '')
    
    # 2. Identify key columns (Target, Feature 1, Feature 2)
    star_col = 'quality_of_patient_care_star_rating'
    mobility_col = 'how_often_patients_got_better_at_walking_or_moving_around'
    cost_col = 'how_much_medicare_spends_on_an_episode_of_care_at_this_agency_compared_to_medicare_spending_across_all_agencies_nationally'
    
    # 3. Convert text numbers to float (handles strings like "Not Available" or "-" as NaN)
    for col in [star_col, mobility_col, cost_col]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    # 4. Implement missing value strategy
    # Drop rows missing the target variable
    df = df.dropna(subset=[star_col])
    
    # Fill clinical and financial features with the mean
    df[mobility_col] = df[mobility_col].fillna(df[mobility_col].mean())
    df[cost_col] = df[cost_col].fillna(df[cost_col].mean())
    
    return df

# Apply the cleaning module
df_clean = clean_cms_data(df_raw.copy())
print("Data preparation and cleaning complete.")

## CRISP-DM Phase 4: Data Modeling
In this phase, I modularize the training process for a Random Forest Regressor to predict the Star Rating based on clinical and financial features.

In [ ]:
def perform_modeling(df):
    """
    Description:
        Prepares features and target, splits the data into training and testing sets, 
        and trains a Random Forest Regressor model.
    
    Arguments:
        df (pandas DataFrame): The cleaned and prepared DataFrame.
        
    Returns:
        model: The trained RandomForestRegressor object.
        X_test: The feature test set.
        y_test: The target test set.
    """
    # Define features and target
    mobility_col = 'how_often_patients_got_better_at_walking_or_moving_around'
    cost_col = 'how_much_medicare_spends_on_an_episode_of_care_at_this_agency_compared_to_medicare_spending_across_all_agencies_nationally'
    
    X = df[[mobility_col, cost_col]]
    y = df['quality_of_patient_care_star_rating']
    
    # Split into train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Initialize and fit model
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    return model, X_test, y_test

# Execute the modeling module
model, X_test, y_test = perform_modeling(df_clean)

## CRISP-DM Phase 5: Evaluation
In this section, I create visualizations to answer my business questions and evaluate the performance of the predictive model.

In [ ]:
def create_visualizations(df, model, X_test, y_test):
    """
    Description:
        Creates bar charts for ownership analysis and outputs model performance metrics.
    
    Arguments:
        df (pandas DataFrame): The cleaned dataset.
        model: The trained machine learning model.
        X_test: Features for evaluation.
        y_test: Actual values for evaluation.
    """
    # Visual 1: Average Star Rating by Ownership
    plt.figure(figsize=(10,5))
    df.groupby('type_of_ownership')['quality_of_patient_care_star_rating'].mean().sort_values().plot(kind='barh', color='teal')
    plt.title('Average Star Rating by Ownership Type')
    plt.show()

    # Visual 2: Medicare Spending by Ownership
    cost_col = 'how_much_medicare_spends_on_an_episode_of_care_at_this_agency_compared_to_medicare_spending_across_all_agencies_nationally'
    plt.figure(figsize=(10,5))
    df.groupby('type_of_ownership')[cost_col].mean().plot(kind='bar', color='orange')
    plt.title('Medicare Spend Ratio (Efficiency) by Ownership')
    plt.show()

    # Model Evaluation Metrics
    preds = model.predict(X_test)
    print(f"Model R2 Score: {r2_score(y_test, preds):.4f}")
    print(f"Mean Squared Error: {mean_squared_error(y_test, preds):.4f}")

# Run evaluation module
create_visualizations(df_clean, model, X_test, y_test)

## CRISP-DM Phase 6: Deployment
The insights from this analysis are deployed via a blog post. Key findings show that non-profit ownership is highly correlated with both higher clinical quality (mobility) and financial efficiency. These results provide a roadmap for patients selecting high-quality home health providers.